# 4.10 Feature Engineering

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.10_feature-engineering.ipynb)

Companion code for Section 4.10 of the book. Run the cells in order from top to bottom.

**Example 4.10.1**

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)
df = pd.DataFrame({
    'income':       np.random.exponential(60000, 1000),   # Right-skewed
    'page_views':   np.random.poisson(50, 1000),          # Count data
    'age':          np.random.normal(40, 12, 1000).clip(18, 80),
    'price':        np.random.exponential(200, 1000),
    'quantity':     np.random.poisson(5, 1000),
})

# ── Log transform - reduce right skew ────────────────────────────
df['log_income']     = np.log1p(df['income'])      # log1p = log(1+x), handles 0
df['log_page_views'] = np.log1p(df['page_views'])

print('Skewness reduction via log transform:')
for raw, log in [('income','log_income'), ('page_views','log_page_views')]:
    print(f'  {raw}: {stats.skew(df[raw]):+.3f} → {raw} log: {stats.skew(df[log]):+.3f}')

# ── Square root - moderate skew reduction ────────────────────────
df['sqrt_income'] = np.sqrt(df['income'])

# ── Box-Cox transform - optimal power transform ───────────────────
df['boxcox_income'], lambda_ = stats.boxcox(df['income'] + 1)
print(f'\nBox-Cox lambda: {lambda_:.4f} (0=log, 1=identity, 0.5=sqrt)')
print(f'Box-Cox skewness: {stats.skew(df.boxcox_income):+.4f}')

# ── Polynomial features - capture non-linear relationships ────────
from sklearn.preprocessing import PolynomialFeatures
X = df[['age','income']].values[:5]
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X)
print(f'\nPolynomial (degree=2) features: {poly.get_feature_names_out(["age","income"])}')
print(f'Shape: {X.shape} → {X_poly.shape}')

# ── Ratio features - often more informative than raw values ───────
df['revenue']        = df['price'] * df['quantity']   # Total revenue
df['revenue_per_view']= df['revenue'] / (df['page_views'] + 1)  # Avoid /0
df['price_to_income'] = df['price'] / (df['income'] + 1)
print(f'\nNew ratio features created: revenue, revenue_per_view, price_to_income')

**Example 4.10.2**

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
# Simulate e-commerce transaction data
n = 5000
dates = pd.date_range('2022-01-01', '2024-01-01', periods=n)
df = pd.DataFrame({
    'transaction_date': dates,
    'amount':  np.random.exponential(150, n),
    'user_id': np.random.randint(1, 500, n)
})

# ── Extract datetime components ───────────────────────────────────
df['year']          = df['transaction_date'].dt.year
df['month']         = df['transaction_date'].dt.month
df['day']           = df['transaction_date'].dt.day
df['day_of_week']   = df['transaction_date'].dt.dayofweek   # 0=Monday
df['day_of_year']   = df['transaction_date'].dt.dayofyear
df['week_of_year']  = df['transaction_date'].dt.isocalendar().week.astype(int)
df['quarter']       = df['transaction_date'].dt.quarter
df['hour']          = df['transaction_date'].dt.hour
df['is_weekend']    = (df['day_of_week'] >= 5).astype(int)
df['is_month_start']= df['transaction_date'].dt.is_month_start.astype(int)
df['is_month_end']  = df['transaction_date'].dt.is_month_end.astype(int)

# ── Cyclical encoding - preserve circular nature of time ──────────
# Month 12 and month 1 are adjacent - but numerically 12 >> 1
# Cyclical encoding: sin/cos transformation preserves adjacency
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)

# ── Rolling window features ───────────────────────────────────────
df_sorted = df.sort_values('transaction_date').reset_index(drop=True)
df_sorted['rolling_7d_mean']  = df_sorted['amount'].rolling(7, min_periods=1).mean()
df_sorted['rolling_30d_std']  = df_sorted['amount'].rolling(30, min_periods=1).std()
df_sorted['rolling_7d_max']   = df_sorted['amount'].rolling(7, min_periods=1).max()

# ── Lag features - previous values as predictors ──────────────────
for lag in [1, 3, 7, 14, 30]:
    df_sorted[f'amount_lag_{lag}'] = df_sorted['amount'].shift(lag)

print('Temporal features created:')
temporal_cols = [c for c in df_sorted.columns if c != 'transaction_date']
print(f'  {len(temporal_cols)} features from {1} datetime column')
print(df_sorted[['transaction_date','month','is_weekend','month_sin',
                  'rolling_7d_mean','amount_lag_1']].head(8).round(3))

**Example 4.10.3**

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    'customer_id': np.repeat(range(200), 10),
    'amount':      np.random.exponential(100, 2000),
    'product_cat': np.random.choice(['Electronics','Clothing','Food','Books'], 2000),
    'returned':    np.random.choice([0,1], 2000, p=[0.85, 0.15]),
})

# ── Customer-level aggregation features ───────────────────────────
# These are 'group-level' features - statistics about each customer
customer_features = df.groupby('customer_id').agg(
    n_orders         = ('amount', 'count'),
    total_spend      = ('amount', 'sum'),
    mean_order_value = ('amount', 'mean'),
    max_order_value  = ('amount', 'max'),
    std_order_value  = ('amount', 'std'),
    return_rate      = ('returned', 'mean'),
    n_categories     = ('product_cat', 'nunique'),
).reset_index()

# Derived customer features
customer_features['spend_per_category'] = (
    customer_features['total_spend'] / customer_features['n_categories']
)
customer_features['is_loyal']  = (customer_features['n_orders'] > 12).astype(int)
customer_features['is_churner']= (customer_features['return_rate'] > 0.30).astype(int)

print('Customer-level features:')
print(customer_features.head(5).round(3))

# ── Interaction features ───────────────────────────────────────────
# When two features together are more informative than either alone
df2 = pd.DataFrame({
    'income':    np.random.normal(60000, 20000, 500),
    'debt':      np.random.normal(15000, 8000, 500).clip(0),
    'credit_score': np.random.normal(680, 80, 500).clip(300, 850),
})

# Debt-to-income ratio - fundamental in credit risk
df2['debt_to_income'] = df2['debt'] / (df2['income'] + 1)

# Credit utilisation tier
df2['income_tier'] = pd.qcut(df2['income'], q=4, labels=['Q1','Q2','Q3','Q4'])

# Flag: high debt AND low income - the dangerous combination
df2['high_risk'] = (
    (df2['debt_to_income'] > 0.35) & (df2['credit_score'] < 620)
).astype(int)

print(f'\nHigh-risk customers: {df2.high_risk.sum()} ({df2.high_risk.mean():.1%})')
print('Interaction features:',
      ['debt_to_income','income_tier','high_risk'])